# Домашняя работа 2 — качество данных

## Подготовка

In [1]:
from pathlib import Path
import pandas as pd
from pandas.testing import assert_frame_equal
DATA = Path("../data")
LOCAL = Path("../data")
WORK = Path("../results")
WORK.mkdir(exist_ok=True)
print("pandas", pd.__version__)

pandas 2.2.2


## Независимая загрузка

In [2]:
decisions = pd.read_csv(
    DATA / "decisions.csv", dtype={"lei": "string", "county_code": "string"}
)
export = pd.read_csv(LOCAL / "training_export.csv", dtype="string")
raw_copy = export.copy(deep=True)
print(decisions.shape, export.shape)
print(export.dtypes)
export.head(5)

(6830, 7) (6832, 7)
source_row       string[python]
activity_year    string[python]
lei              string[python]
county_code      string[python]
action_taken     string[python]
loan_amount      string[python]
income           string[python]
dtype: object


,source_row,activity_year,lei,county_code,action_taken,loan_amount,income
0,2,2023,549300VZVN841I2ILS84,11001,1,oops,72.0
1,3,2023,549300P1347BFCBHJV60,11001,88,585000.0,174.0
2,5,2023,549300P1347BFCBHJV60,11001,1,275000.0,BAD
3,7,2023,549300RPOGWJRH63HS39,11001,1,295000.0,63.0
4,8,2023,549300RPOGWJRH63HS39,11001,1,1125000.0,247.0


## H1. Собственная функция очистки

In [6]:
def clean_export(source):
    data = source.copy(deep=True)

    audit = {
        "input": len(data),
        "replays": 0,
        "clean": 0,
        "quarantine": 0,
        "parse_errors": 0
    }

    known_rows = set(decisions["source_row"].dropna())

    data["county_code"] = data["county_code"].astype("string").str.strip()

    for col in ["source_row", "action_taken", "loan_amount", "income"]:
        old = data[col].copy()
        data[col] = pd.to_numeric(data[col], errors="coerce")
        audit["parse_errors"] += int((old.notna() & data[col].isna()).sum())

    data["source_row"] = data["source_row"].astype("Int64")
    data["action_taken"] = data["action_taken"].astype("Int64")

    duplicated_rows = data.duplicated(keep="first")
    audit["replays"] = int(duplicated_rows.sum())
    data = data.loc[~duplicated_rows].copy()

    versions = data.groupby("source_row", dropna=False).size()
    conflicts = versions[versions > 1].index
    assert len(conflicts) == 0, "Обнаружены конфликтующие версии source_row"

    valid = (
        data["source_row"].isin(known_rows)
        & data["action_taken"].isin([1, 2, 3])
        & data["loan_amount"].notna()
        & (data["loan_amount"] > 0)
        & (data["county_code"] == "11001")
    )

    quarantine = data.loc[~valid].copy()
    clean = data.loc[valid].copy()

    quarantine["reason"] = ""

    quarantine.loc[~quarantine["source_row"].isin(known_rows), "reason"] += "unknown_source_row; "
    quarantine.loc[~quarantine["action_taken"].isin([1, 2, 3]), "reason"] += "invalid_action_taken; "
    quarantine.loc[quarantine["loan_amount"].isna(), "reason"] += "missing_loan_amount; "
    quarantine.loc[quarantine["loan_amount"].notna() & (quarantine["loan_amount"] <= 0), "reason"] += "invalid_loan_amount; "
    quarantine.loc[quarantine["county_code"] != "11001", "reason"] += "invalid_county_code; "

    audit["clean"] = len(clean)
    audit["quarantine"] = len(quarantine)

    return clean, quarantine, audit

## H2. Год и известный доход

In [4]:
export_before = export.copy(deep=True)

clean, quarantine, audit = clean_export(export)

print(audit)
print(clean.head())
print(quarantine.head())

assert export.equals(export_before)
assert clean["activity_year"].astype(str).eq("2023").all()

print("Количество чистых строк:", len(clean))

known_income = clean["income"].notna().sum()
missing_income = clean["income"].isna().sum()
mean_income = clean["income"].mean()

print("Известный доход:", known_income)
print("Пропущенный доход:", missing_income)
print("Средний доход:", mean_income)

assert audit["input"] == audit["replays"] + audit["clean"] + audit["quarantine"]

{'input': 6832, 'replays': 2, 'clean': 6828, 'quarantine': 2, 'parse_errors': 2}
   source_row activity_year                   lei county_code  action_taken  \
2           5          2023  549300P1347BFCBHJV60       11001             1   
3           7          2023  549300RPOGWJRH63HS39       11001             1   
4           8          2023  549300RPOGWJRH63HS39       11001             1   
5           9          2023  549300RPOGWJRH63HS39       11001             1   
6          10          2023  549300RPOGWJRH63HS39       11001             1   

   loan_amount  income  
2     275000.0    <NA>  
3     295000.0    63.0  
4    1125000.0   247.0  
5     305000.0   152.0  
6     505000.0   110.0  
   source_row activity_year                   lei county_code  action_taken  \
0           2          2023  549300VZVN841I2ILS84       11001             1   
1           3          2023  549300P1347BFCBHJV60       11001            88   

   loan_amount  income                  reason  
0      

## H3. Проверка отчётного года

In [5]:
wrong_year = export.copy(deep=True)
wrong_year.loc[wrong_year.index[0], "activity_year"] = "2022"

year_ok = wrong_year["activity_year"].eq("2023").fillna(False).all()

print("Все записи относятся к 2023 году:", year_ok)

assert year_ok == False
assert export.equals(export_before)

Все записи относятся к 2023 году: False


## H+1. Конфликт исходных значений

## H+2. Совпадение опубликованных полей